In [1]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer
from tflite_inference_input_adapter import TFLiteInferenceWithInputAdapter

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

TensorFlow version:  2.17.0


In [27]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_3_margin_15_rq_wb_alif_blurred"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = " /home/klymy/data/cam4_revB/yolo_dataset/batch1_3_margin_15_wb_alif_blurred/project_holdout.yaml"
POC_PTH = "/home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/project_pg.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

In [28]:
model = YOLO(WEIGHTS_PTH)

In [29]:
inferer = TFLiteInferer(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

inferer_adapter = TFLiteInferenceWithInputAdapter(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
    head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
    adapter_pth="../input_adapter/input_adapter_full_integer_quant.tflite",
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

In [30]:
args["data"] = HOLDOUT_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
    False,
)

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_3_margin_15_wb_alif_blurred/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:05<00:00,  2.91s/it]


                   all        148        409      0.828      0.768      0.854      0.631      0.839      0.778      0.878      0.875
   task    F1     P     R  MAP@50
0   box 0.797 0.828 0.768   0.854
1  pose 0.807 0.839 0.778   0.878


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_3_margin_15_wb_alif_blurred/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.23s/it]


                   all        148        409      0.703      0.736      0.753      0.542      0.727      0.796      0.811      0.802
   task    F1     P     R  MAP@50
0   box 0.719 0.703 0.736   0.753
1  pose 0.760 0.727 0.796   0.811


In [31]:
args["data"] = POC_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer_adapter, args),
    False,
)

val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:07<00:00,  3.75s/it]


                   all        191        417      0.617      0.707      0.723      0.513      0.616      0.779      0.772      0.744
   task    F1     P     R  MAP@50
0   box 0.659 0.617 0.707   0.723
1  pose 0.688 0.616 0.779   0.772


val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]
tst validation: 100%|██████████| 2/2 [00:08<00:00,  4.17s/it]


                   all        191        417      0.724      0.659      0.764      0.541      0.755      0.688      0.797      0.781
   task    F1     P     R  MAP@50
0   box 0.690 0.724 0.659   0.764
1  pose 0.720 0.755 0.688   0.797
